In [ ]:
# ============================================================
# Pipeline stage : Modeling / Inference (nearest-neighbor retrieval)
#
# What this snippet teaches:
#   How to use a KD-Tree to find the k nearest items to a query
#   point quickly, instead of comparing against every item.
#   Build the tree ONCE (offline), then query it MANY times (online).
#
# Summary of the lesson (in comments):
#   - KD-Tree splits space by the median, one feature per level,
#     cycling through features -> balanced tree, depth ~ log2(n).
#   - Query = go down to a leaf, then walk back up and prune
#     branches that cannot contain anything closer.
#   - Fast only in LOW dimensions (roughly <= 20 features).
#   - Distances matter -> features MUST be on the same scale.
# ============================================================

import joblib
import numpy as np
from sklearn.neighbors import KDTree
from sklearn.preprocessing import StandardScaler

# ---------- 1) Load data ----------
# Example: restaurants with a few numeric features
# (lat, lon, avg_price, rating)  -> 4 dimensions, KD-Tree is a good fit
df = ...  # TODO: load your DataFrame (e.g. pd.read_csv(...))
FEATURES = ["lat", "lon", "avg_price", "rating"]  # TODO: adjust to your data

X = df[FEATURES].to_numpy()

# ---------- 2) Scale features ----------
# KD-Tree uses distance. Without scaling, avg_price (thousands)
# would completely drown out rating (0-5).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------- 3) Build the tree once ----------
tree = KDTree(X_scaled, leaf_size=30, metric="euclidean")

# Persist both: the tree AND the scaler (they must stay in sync)
joblib.dump({"tree": tree, "scaler": scaler}, "artifacts/kdtree.joblib")

# ---------- 4) Query (this is what runs per request) ----------
def find_similar(query_row: np.ndarray, k: int = 5):
    """Return the k nearest rows of df to query_row."""
    q = scaler.transform(query_row.reshape(1, -1))   # same scaling as training
    dist, idx = tree.query(q, k=k)
    return df.iloc[idx[0]].assign(distance=dist[0])

# TODO: replace with a real query coming from your API / UI
sample_query = np.array([35.70, 51.40, 250_000, 4.5])
print(find_similar(sample_query, k=5))

# ============================================================
# Engineering red flags (read before shipping!)
#   1. Forgot to scale the features        -> results are dominated by one big-valued column.
#   2. Scaler fitted on query data         -> always fit on train, only transform at query time.
#   3. Too many features (say 50+)         -> KD-Tree degrades to brute force. Check Ball Tree / FAISS.
#   4. NaN values in X                     -> KDTree raises an error. Impute before building.
#   5. Categorical columns encoded as ints -> distance becomes meaningless. Encode properly.
#   6. Data changes often                  -> the tree is static in sklearn; you must rebuild it.
#   7. Lat/lon with euclidean distance     -> only OK for small areas. Use haversine (BallTree) for real geo.
#   8. Never benchmark against brute force -> on small data, brute force may be faster. Measure.
# ============================================================